<a href="https://colab.research.google.com/github/wgcv/gesture-mlai-assessment/blob/main/submission/gesture_mlai_assessment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gesture — AI/ML Engineer Assessment

## Data analysis

### Download CSV

In [1]:
import requests

urls = [
    "https://raw.githubusercontent.com/wgcv/gesture-mlai-assessment/refs/heads/main/data/user_events.csv",
    "https://raw.githubusercontent.com/wgcv/gesture-mlai-assessment/refs/heads/main/data/product_catalog.csv"
]

for url in urls:
    filename = url.split("/")[-1]

    try:
        r = requests.get(url)
        r.raise_for_status()

        with open(filename, "wb") as f:
            f.write(r.content)

        print(f"Downloaded: {filename}")

    except requests.exceptions.RequestException as e:
        print(f"Failed: {url} -> {e}")

Downloaded: user_events.csv
Downloaded: product_catalog.csv


### Load user_events

In [4]:
import pandas as pd

df_events = pd.read_csv("user_events.csv")

df_events.head()


,event_id,user_id,event_type,timestamp,sku_id,price_usd,recipient_id,platform,user_type
0,evt_0018094,usr_0556,app_open,2026-01-01 09:57:00,NaN,NaN,NaN,ios,b2b
1,evt_0009755,usr_0302,app_open,2026-02-18 08:57:00,NaN,NaN,NaN,android,b2b
2,evt_0021240,usr_0649,app_open,2025-11-30 22:03:00,NaN,NaN,NaN,ios,b2c
3,evt_0006725,usr_0209,product_view,2026-03-28 15:26:00,SKU029,82.0,NaN,ios,b2c
4,evt_0008882,usr_0285,app_open,2026-03-29 07:03:00,NaN,NaN,NaN,ios,b2c


### Load product_catalog

In [2]:
import pandas as pd

df_products = pd.read_csv("product_catalog.csv")

df_products.head()


,sku_id,title,category,price_usd,description
0,SKU001,Luxury Candle Set,candle,65,Hand-poured soy wax candles in cedarwood and v...
1,SKU002,Artisan Chocolate Box,food,45,24-piece single-origin chocolate assortment. D...
2,SKU003,Silk Sleep Mask,wellness,38,"100% mulberry silk, adjustable strap, blocks l..."
3,SKU004,Personalized Journal,stationery,55,Leather-bound journal with custom name embossi...
4,SKU005,Premium Tea Collection,food,42,"Curated set of 12 loose-leaf teas from Japan, ..."


In [5]:
df = df_events.merge(
    df_products,
    on="sku_id",
    how="left",
    suffixes=("_event", "_product")
)

In [6]:
df

,event_id,user_id,event_type,timestamp,sku_id,price_usd_event,recipient_id,platform,user_type,title,category,price_usd_product,description
0,evt_0018094,usr_0556,app_open,2026-01-01 09:57:00,NaN,NaN,NaN,ios,b2b,NaN,NaN,NaN,NaN
1,evt_0009755,usr_0302,app_open,2026-02-18 08:57:00,NaN,NaN,NaN,android,b2b,NaN,NaN,NaN,NaN
2,evt_0021240,usr_0649,app_open,2025-11-30 22:03:00,NaN,NaN,NaN,ios,b2c,NaN,NaN,NaN,NaN
3,evt_0006725,usr_0209,product_view,2026-03-28 15:26:00,SKU029,82.0,NaN,ios,b2c,Silver Chain Bracelet,jewelry,82.0,"Sterling silver, 7-inch adjustable chain. Lobs..."
4,evt_0008882,usr_0285,app_open,2026-03-29 07:03:00,NaN,NaN,NaN,ios,b2c,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
26769,evt_0021575,usr_0663,product_view,2025-12-14 21:31:00,SKU006,49.0,NaN,ios,b2c,Wireless Charging Pad,tech,49.0,"15W fast-charge Qi pad, works with all Qi-enab..."
26770,evt_0005390,usr_0162,product_view,2025-12-25 06:51:00,SKU030,46.0,NaN,android,b2c,Facial Roller Set,beauty,46.0,Rose quartz roller and gua sha tool. Comes wit...
26771,evt_0000860,usr_0025,app_open,2026-02-24 09:50:00,NaN,NaN,NaN,android,b2b,NaN,NaN,NaN,NaN
26772,evt_0015795,usr_0484,add_to_cart,2025-11-29 17:25:00,SKU023,35.0,NaN,android,b2c,Desk Succulent Trio,plants,35.0,Three 2-inch succulents in matching white cera...


### Data Exploration

In [7]:
print(df.describe())
print("------------")
print(df.info())
print("------------")
print(df.describe())

       price_usd_event  price_usd_product
count     14720.000000       14720.000000
mean         58.229552          58.293003
std          17.731044          17.640843
min          -1.000000          32.000000
25%          46.000000          46.000000
50%          55.000000          55.000000
75%          68.000000          68.000000
max         120.000000         120.000000
------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26774 entries, 0 to 26773
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   event_id           26774 non-null  object 
 1   user_id            26774 non-null  object 
 2   event_type         26774 non-null  object 
 3   timestamp          26239 non-null  object 
 4   sku_id             14720 non-null  object 
 5   price_usd_event    14720 non-null  float64
 6   recipient_id       1325 non-null   object 
 7   platform           26774 non-null  object 
 8   user_type      

In [8]:
df["event_type"].unique() # Unique values on event_type

array(['app_open', 'product_view', 'add_to_cart', 'purchase', 'gift_send'],
      dtype=object)

In [9]:
df[
    df["price_usd_event"].fillna(-1) != df["price_usd_product"].fillna(-1)
]

,event_id,user_id,event_type,timestamp,sku_id,price_usd_event,recipient_id,platform,user_type,title,category,price_usd_product,description
232,evt_0018671,usr_0575,product_view,2025-11-30 10:09:00,SKU030,-1.0,NaN,web,b2b,Facial Roller Set,beauty,46.0,Rose quartz roller and gua sha tool. Comes wit...
3052,evt_0015283,usr_0467,add_to_cart,2025-10-13 13:50:00,SKU004,-1.0,NaN,web,b2c,Personalized Journal,stationery,55.0,Leather-bound journal with custom name embossi...
4207,evt_0007932,usr_0254,product_view,2025-11-23 07:51:00,SKU015,-1.0,NaN,ios,b2b,Ceramic Mug Set,home,52.0,"Set of 2 hand-thrown ceramic mugs, 14oz. Dishw..."
5271,evt_0014958,usr_0462,add_to_cart,2026-03-18 14:16:00,SKU018,-1.0,NaN,android,b2c,Yoga Mat & Block Set,wellness,78.0,"Non-slip natural rubber mat (6mm), cork block,..."
5570,evt_0006538,usr_0199,product_view,2026-01-08 15:34:00,SKU022,-1.0,NaN,android,b2c,Silk Pillowcase Set,home,72.0,Two queen-size 22-momme silk pillowcases. Redu...
5608,evt_0004420,usr_0123,add_to_cart,2025-11-13 05:48:00,SKU012,-1.0,NaN,ios,b2b,Leather Card Holder,accessories,48.0,"Full-grain leather, holds 8 cards, slim profil..."
6241,evt_0015350,usr_0467,product_view,2026-03-10 23:49:00,SKU030,-1.0,NaN,web,b2c,Facial Roller Set,beauty,46.0,Rose quartz roller and gua sha tool. Comes wit...
9544,evt_0006543,usr_0199,product_view,2025-12-09 19:10:00,SKU011,-1.0,NaN,android,b2c,Knit Throw Blanket,home,75.0,"100% merino wool, 50x60 inches, chunky knit. A..."
11505,evt_0016352,usr_0498,gift_send,2026-03-19 21:23:00,SKU017,-1.0,rec_0031,web,b2c,Monogram Tote Bag,accessories,68.0,Canvas tote with custom monogram. Interior poc...
20412,evt_0023218,usr_0701,purchase,2026-02-10 00:14:00,SKU006,-1.0,NaN,ios,b2c,Wireless Charging Pad,tech,49.0,"15W fast-charge Qi pad, works with all Qi-enab..."


In [10]:
## Check missing price
df_filtered = df[
    df["price_usd_event"].isna() &
    (df["event_type"] != "app_open")]

#### Fix missing price

In [12]:
df["price_usd_event"] = df["price_usd_event"].where(
    df["price_usd_event"] >= 0,
    df["price_usd_product"]
)

In [13]:
df[
    df["price_usd_event"].fillna(-1) != df["price_usd_product"].fillna(-1)
]

,event_id,user_id,event_type,timestamp,sku_id,price_usd_event,recipient_id,platform,user_type,title,category,price_usd_product,description


#### Remove Wrong Timestamp

In [14]:
df["timestamp_parsed"] = pd.to_datetime(
    df["timestamp"],
    errors="coerce"
)

In [15]:
df_bad = df[
    df["timestamp"].isna() | df["timestamp_parsed"].isna()
]

In [16]:
df_bad

,event_id,user_id,event_type,timestamp,sku_id,price_usd_event,recipient_id,platform,user_type,title,category,price_usd_product,description,timestamp_parsed
19,evt_0018686,usr_0575,purchase,NaN,SKU006,49.0,NaN,web,b2b,Wireless Charging Pad,tech,49.0,"15W fast-charge Qi pad, works with all Qi-enab...",NaT
78,evt_0024667,usr_0736,purchase,NaN,SKU006,49.0,NaN,ios,b2c,Wireless Charging Pad,tech,49.0,"15W fast-charge Qi pad, works with all Qi-enab...",NaT
152,evt_0019920,usr_0621,app_open,NaN,NaN,NaN,NaN,ios,b2c,NaN,NaN,NaN,NaN,NaT
165,evt_0000866,usr_0025,app_open,NaN,NaN,NaN,NaN,android,b2b,NaN,NaN,NaN,NaN,NaT
188,evt_0026264,usr_0790,product_view,NaN,SKU008,55.0,NaN,ios,b2c,Succulent Arrangement,plants,55.0,5 hand-selected succulents in a ceramic plante...,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26488,evt_0008680,usr_0282,app_open,NaN,NaN,NaN,NaN,ios,b2c,NaN,NaN,NaN,NaN,NaT
26610,evt_0015934,usr_0487,app_open,NaN,NaN,NaN,NaN,ios,b2c,NaN,NaN,NaN,NaN,NaT
26645,evt_0019483,usr_0602,app_open,NaN,NaN,NaN,NaN,ios,b2c,NaN,NaN,NaN,NaN,NaT
26722,evt_0010555,usr_0325,product_view,NaN,SKU024,36.0,NaN,android,b2c,Artisan Hot Sauce Set,food,36.0,Four small-batch hot sauces ranging from mild ...,NaT


In [17]:
df = df[df["timestamp_parsed"].notna()]

In [18]:
def get_feature_columns(features_df):
    """
    Retorna lista de columnas de features (excluye user_id, snapshot_date, target).
    """
    exclude_cols = ['user_id', 'snapshot_date', 'target']
    return [col for col in features_df.columns if col not in exclude_cols]
